In [7]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import regionmask
import os
import sys

import geopandas as gp
import rioxarray
from shapely.geometry import mapping


import warnings
warnings.filterwarnings("ignore")
!conda list netcdf4

# packages in environment at /home/aelyoussoufi/miniconda3/envs/latest:
#
# Name                    Version                   Build  Channel
netcdf4                   1.7.2           nompi_py310h5146f0f_101    conda-forge


In [2]:
def detrend_dim(da, dim, deg=1):
    # detrend along a single dimension
    p = da.polyfit(dim=dim, deg=deg)
    fit = xr.polyval(da[dim], p.polyfit_coefficients)
    return da - fit

In [3]:
def init_climregions():
    se_dict={'name':'Southeast','short_name':'SE',
             'states':['AL','FL','GA','NC','SC','VA']}
    ne_dict={'name':'Northeast','short_name':'NE',
             'states':['CT','DE','ME','MD','MA','NH','NJ','NY','PA','RI','VT']}
    umidwest_dict={'name':'Upper Midwest','short_name':'UMW',
             'states':['IA','MI','MN','WI']}
    ohiov_dict={'name':'Ohio Valley','short_name':'OHV',
                'states':['IL','IN','KY','MI','MO','OH','TN','WV']}
    nrockies_dict={'name':'Northern Rockies and Plains','short_name':'NRP',
                   'states':['MT','NE','ND','SD','WY']}
    south_dict={'name':'South','short_name':'SOUTH',
                   'states':['AR','KS','LA','MS','OK','TX']}
    sw_dict={'name':'Southwest','short_name':'SW',
                   'states':['AZ','CO','NM','UT']}
    nw_dict={'name':'Northwest','short_name':'NW',
                   'states':['ID','OR','WA']}
    west_dict={'name':'West','short_name':'WEST',
                'states':['CA','NV']}
    


    regs_dict=[se_dict,ne_dict,umidwest_dict,ohiov_dict,nrockies_dict,south_dict,sw_dict,nw_dict,west_dict]
    
    return regs_dict


def daily_climo(da,varname,**kwargs):
  
    # This function is adapted the code written by Ray Bell for the SubX project.  
    # It calculates climatology following the SubX smoothed and periodic climatology method.
    
    clim_fname = kwargs.get('fname', None)
    
    # Average daily data
    da_day_clim = da.groupby('time.dayofyear').mean('time')
    
    # Rechunk for time
    da_day_clim = da_day_clim.chunk({'dayofyear': 366})
    
    
    # Pad the daily climatolgy with nans
    x = np.empty((366, len(da_day_clim.lat), len(da_day_clim.lon)))
    x.fill(np.nan)
    _da = xr.DataArray(x,name=varname, coords=[np.linspace(1, 366, num=366, dtype=np.int64),
                              da_day_clim.lat, da_day_clim.lon],
                              dims = da_day_clim.dims)
    da_day_clim_wnan = da_day_clim.combine_first(_da)

    
    # Period rolling twice to make it triangular smoothing
    # See https://bit.ly/2H3o0Mf
    da_day_clim_smooth = da_day_clim_wnan.copy()
 
    

    for i in range(2):
        # Extand the DataArray to allow rolling to do periodic
        da_day_clim_smooth = xr.concat([da_day_clim_smooth[-15:],
                                        da_day_clim_smooth,
                                        da_day_clim_smooth[:15]],
                                        'dayofyear')
        # Rolling mean
        da_day_clim_smooth = da_day_clim_smooth.rolling(dayofyear=31,
                                                        center=True,
                                                        min_periods=1).mean()
        # Drop the periodic boundaries
        da_day_clim_smooth = da_day_clim_smooth.isel(dayofyear=slice(15, -15))

    
    # Extract the original days
    da_day_clim_smooth = da_day_clim_smooth.sel(dayofyear=da_day_clim.dayofyear)

    da_day_clim_smooth.name=varname
    ds_day_clim_smooth=da_day_clim_smooth.to_dataset()
    
    # Save to file if filename provide and return True, otherwise return the data
    if (clim_fname):
        ds_day_clim_smooth.to_netcdf(clim_fname)
        return True
    else:
        return ds_day_clim_smooth

In [4]:
baseDir='/data/esplab/aelyoussoufi/'
dataset='snow_water_equivalent'
varname='SWE'
files=baseDir+'snow_water_equivalent.nc'  # File in current directory
states_shapefile='cb_2018_us_state_20m.shp'  # Shapefile in current directory
outPath = '.' 
regs_dict=init_climregions()

In [5]:
files

'/data/esplab/aelyoussoufi/snow_water_equivalent.nc'

In [6]:
ds=xr.open_mfdataset(files)
ds

ValueError: did not find a match in any of xarray's currently installed IO backends ['netcdf4', 'scipy', 'rasterio']. Consider explicitly selecting one of the installed engines via the ``engine`` parameter, or installing additional IO dependencies, see:
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html
https://docs.xarray.dev/en/stable/user-guide/io.html

In [ ]:
ds = ds.assign_coords(longitude=(((ds.X + 180) % 360) - 180)).sortby('X')

In [ ]:
states = gp.read_file(states_shapefile)

In [ ]:
mask = regionmask.mask_geopandas(states, ds['X'], ds['Y'])
ds['mask']=mask
ds

In [ ]:
ds_us=ds.sel(Y=slice(25,50),X=slice(-125,-50)).compute()

In [ ]:
ds_us=ds_us.rename({'Y':'lat','X':'lon'})

In [ ]:
for state_num,abbrv in zip(states.index.values,states['STUSPS'].values):

    print(abbrv)
    
    ds_state=ds_us.where(ds_us['mask']==state_num,drop=True).compute()
    
    if (len(ds_state['lat'])>0 and len(ds_state['lon']>0)):
    
        ds_climo=daily_climo(ds_state['snow_water_equivalent'],'snow_water_equivalent')['snow_water_equivalent'].chunk({'dayofyear':366,'lat':'auto','lon':'auto'})
        ds_anoms=ds_state['snow_water_equivalent'].groupby('time.dayofyear')-ds_climo
        outfile='../swe.2003-01-01.2020-12-31.'+abbrv+'.nc'
        ds_anoms.to_netcdf(outfile)